# URAE Tutorial: Parametric CAD & Isogeometric Analysis (IGA)

This tutorial explores URAE's integrated CAD and finite element simulation engine:
- **Parametric Machinery Generators**: Involute gears, metric bolts, NACA wings, and coil springs
- **Exact B-Spline Basis**: Recursive Cox-de Boor evaluation $N_{i, p}(u)$ and exact analytical derivatives $N'_{i, p}(u)$
- **N-Dimensional NURBS**: Projective weighted control nets and spatial surface mappings $\mathbf{S}(u, v)$
- **Isogeometric Analysis (IGA)**: Direct stiffness matrix quadrature $K_{ab} = \int_\Omega \nabla R_a \cdot \nabla R_b \, d\Omega$ preserving exact CAD geometry

In [1]:
import urae

# Generate a precision involute spur gear CAD mesh
gear_mesh = urae.cad.generate_involute_gear(
    teeth=20,
    module=2.0,
    pressure_angle=20.0,
    face_width=10.0
)
print("Involute Gear Mesh: Generated", len(gear_mesh.nodes), "nodes and", len(gear_mesh.elements), "faces.")

In [2]:
# Generate an ISO Metric M8 Threaded Bolt
bolt_mesh = urae.cad.generate_iso_screw(
    diameter=8.0,
    pitch=1.25,
    length=25.0,
    head_type="Hex"
)
print("ISO Bolt Mesh: Generated", len(bolt_mesh.nodes), "nodes.")

In [3]:
# Construct a quadratic 2D NURBS surface patch
knots_u = [0.0, 0.0, 0.0, 1.0, 1.0, 1.0]
knots_v = [0.0, 0.0, 0.0, 1.0, 1.0, 1.0]

# Control net: 3x3 array of weighted control points
control_points = [
    urae.iga.WeightedControlPoint([0.0, 0.0, 0.0], 1.0),
    urae.iga.WeightedControlPoint([0.0, 1.0, 0.5], 0.707),
    urae.iga.WeightedControlPoint([0.0, 2.0, 0.0], 1.0),
    urae.iga.WeightedControlPoint([1.0, 0.0, 0.5], 0.707),
    urae.iga.WeightedControlPoint([1.0, 1.0, 1.0], 1.0),
    urae.iga.WeightedControlPoint([1.0, 2.0, 0.5], 0.707),
    urae.iga.WeightedControlPoint([2.0, 0.0, 0.0], 1.0),
    urae.iga.WeightedControlPoint([2.0, 1.0, 0.5], 0.707),
    urae.iga.WeightedControlPoint([2.0, 2.0, 0.0], 1.0),
]

patch = urae.iga.NurbsPatchND.surface_2d(2, 2, knots_u, knots_v, 3, 3, control_points)

# Evaluate midpoint and spatial metric Jacobian determinant
pt_mid = patch.evaluate_surface(0.5, 0.5)
det_j = patch.metric_tensor_determinant(0.5, 0.5)
print("NURBS Surface Point at (0.5, 0.5):", pt_mid)
print("Surface Area Metric |det(J)|:", det_j)

In [4]:
# Directly assemble an Isogeometric Analysis (IGA) stiffness matrix entry
k_00 = urae.iga.IgaStiffnessMatrix.compute_entry_2d(
    basis_idx_a=0,
    basis_idx_b=0,
    degree_p=2,
    degree_q=2,
    knots_u=knots_u,
    knots_v=knots_v
)
print("IGA Stiffness Matrix Entry K_{0,0} =", k_00)